# CNN Classification Model for Plant Leaf Diseases

This notebook implements the Convolutional Neural Network model for classifying 39 distinct crop species and their diseases based on the image datasets.

## Step 1: Import Libraries and GPU Setup
Installs dependencies and enables maximum GPU efficiency via memory growth.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

def setup_gpu():
    """Configures TensorFlow to use maximum GPU efficiency."""
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for gpu in physical_devices:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✅ GPU successfully configured. Found {len(physical_devices)} GPUs.")
        except RuntimeError as e:
            print("⚠️ GPU Setup Error:", e)
    else:
        print("⚠️ No GPU found. Training will use CPU, which may be significantly slower.")

setup_gpu()

## Step 2: Data Preprocessing (80/20 Split)
We use `ImageDataGenerator` to load the dataset, normalize image values, and split the data into 80% Training and 20% Validation subsets.

In [ ]:
DATASET_DIR = '../Original/'

datagen = ImageDataGenerator(
    rescale=1./255,      # Normalize all pixel intensities to [0, 1]
    validation_split=0.2 # 20% validation split
)

print("Loading Training data:")
train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

print("Loading Validation data:")
validation_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

## Step 3 & 4: Convolutional Neural Network Architecture
Here we define the deep CNN using 3 Convolutional Blocks followed by Flattening and a Dense network. The final output layer has 39 endpoints reflecting our specific categorical classes.

In [ ]:
def build_cnn(input_shape=(256, 256, 3), num_classes=39):
    """Defines a deep CNN architecture capable of classifying plant species and disease."""
    model = models.Sequential([
        # Convolutional Block 1 - Extracts foundational edges and textures
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        
        # Convolutional Block 2 - Extracts specific shapes and spotting/mold visuals
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Convolutional Block 3 - Extracts complex high-level disease characteristics
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flatten feature maps into a 1D tensor
        layers.Flatten(),
        
        # Fully Connected (Dense) Layers for mathematical reasoning
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        
        # Output layer with exactly 39 endpoints
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Compile the final layout
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

model = build_cnn()
model.summary()

## Step 5: Training the Model
Run the following block to train the CNN using our generators over several epochs.

In [ ]:
# Uncomment and run to begin training:
# history = model.fit(
#     train_generator,
#     validation_data=validation_generator,
#     epochs=10
# )